# Helper functions (and examples) to align (and view) molecules

In [1]:
from pymatgen.core import Molecule
from pymatgen.analysis import molecule_matcher

def align(base_atoms, target_atoms, threshold=0.5, verbose=False):
    if verbose: print('Convert ASE atoms to Pymatgen molecule')
    base = Molecule.from_ase_atoms(base_atoms)
    target = Molecule.from_ase_atoms(target_atoms)
    
    # Get initial guess
    if verbose: print('Computing initial guess via cheap and naive alignment')
    matcher_guess = molecule_matcher.HungarianOrderMatcher(target)
    aligned_guess, rmsd_guess = matcher_guess.fit(base)
    if verbose: print(f'Hungarian-based alignment yielded: iRMSD={rmsd_guess}')
    
    # Exact matcher
    effective_threshold = min(rmsd_guess+1e-3, threshold)
    if verbose: print(f'Attempting exact match with iRMSD threshold: {effective_threshold}')
    matcher = molecule_matcher.GeneticOrderMatcher(target, threshold=effective_threshold)
    results = matcher.fit(aligned_guess)
    
    # Return best result
    if not results:
        if verbose: print(f'Could not find alignment below iRMSD threshold')
        return(aligned_guess.to_ase_atoms(), None)
    aligned, rmsd = min(results, key=lambda x:x[-1])
    if verbose: print(f'Best alignment found with iRMSD={rmsd}')
    return(aligned.to_ase_atoms(), rmsd)

In [2]:
import ase.io

def write_all_aligned(input_path, output_path, index_target=0, threshold=0.5, verbose=False):
    if verbose: print(f'Reading geometries from {input_path}')
    all_atoms = ase.io.read(input_path, index=':')

    if verbose: print(f'Aligning all {len(all_atoms)} geometries found, w.r.t. to geometry {index_target}, using threshold {threshold}')
    target = all_atoms[index_target]
    all_comments = []
    for current_index, atoms in enumerate(all_atoms):
        # Extract comments
        comments = ' '.join(atoms.info)
        
        # Skip if reference geometry
        if current_index == index_target:
            all_comments.append(comments)
            continue

        # Compute and save aligned geometry
        aligned, rmsd = align(atoms, target, threshold=threshold, verbose=verbose)
        all_atoms[current_index] = aligned
        all_comments.append(f'{comments} (iRMSD={rmsd})')

    if verbose: print(f'Writing all {len(all_atoms)} aligned geometries to {output_path}')
    for current_index, (atoms, comment) in enumerate(zip(all_atoms, all_comments)):
        ase.io.write(output_path, atoms, comment=comment, append=(current_index > 0))

In [3]:
# conda install -c conda-forge ipywidgets nglview
from ase.visualize import view

def view_aligned(aligned, target):
    v = view(aligned+target, viewer='ngl')
    v.view.remove_spacefill()
    v.view.add_ball_and_stick(selection=range(len(aligned)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.2, radiusType='covalent')
    v.view.add_ball_and_stick(selection=range(len(aligned),len(aligned)+len(target)), cylinderOnly=False, aspectRatio=3.0, radiusScale=0.1, radiusType='covalent')
    return(v)

# Example on single pair for molecules to align and view

In [4]:
import ase.io

all_atoms = ase.io.read('results/sample_selected_TS-20250703-SCAN-6_7-w_wo_selfloops-lr2_5_lr5_0e-4-ValFix3/rcmconly_passerini-TS13605.xyz', index=':')
target = all_atoms[1]
base = all_atoms[-1]
' '.join(base.info)

'Generated/inpainted transition state. RMSD: 0.343857 Å. By model /misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/8w-lr5e-4-ValFix3-SCAN-leftnetefadd8bd15e7/ddpm-epoch'

In [5]:
aligned_base, rmsd = align(base, target, threshold=0.5, verbose=True)
print(f'iRSMD={rmsd}')

Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.9947714639023563
Attempting exact match with iRMSD threshold: 0.5
Best alignment found with iRMSD=0.34385699674955833
iRSMD=0.34385699674955833


In [6]:
view_aligned(aligned_base, target)

# Example to produce aligned XYZs

In [7]:
input_path = 'results/sample_selected_TS-20250703-SCAN-6_7-w_wo_selfloops-lr2_5_lr5_0e-4-ValFix3/rcmconly_passerini-TS13605.xyz'
output_path = input_path.removesuffix('.xyz') + '_aligned.xyz'

write_all_aligned(input_path, output_path, index_target=1, threshold=0.5, verbose=True)

Reading geometries from results/sample_selected_TS-20250703-SCAN-6_7-w_wo_selfloops-lr2_5_lr5_0e-4-ValFix3/rcmconly_passerini-TS13605.xyz
Aligning all 16 geometries found, w.r.t. to geometry 1, using threshold 0.5
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.16246073623412205
Attempting exact match with iRMSD threshold: 0.16346073623412205
Best alignment found with iRMSD=0.1605661747951376
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=0.3407122497455898
Attempting exact match with iRMSD threshold: 0.3417122497455898
Best alignment found with iRMSD=0.3379527791611744
Convert ASE atoms to Pymatgen molecule
Computing initial guess via cheap and naive alignment
Hungarian-based alignment yielded: iRMSD=1.0164275901520556
Attempting exact match with iRMSD threshold: 0.5
Could not find alignment below iRMSD threshold

# Trond: align the generated multi_block XYZs 
Ground truth expected as second block, i.e., target_index=1. This is because the blocks are expected to be ordered as:
```
Reference reactant
Reference transition state
Reference product
Generated transition state 1
Generated transition state 2
...
```

In [8]:
import os

def align_many_files(input_dir: str, output_dir: str=None, index_target=1, threshold=0.5, verbose=True):
    if output_dir is not None:
        output_dir = input_dir
    files = os.listdir(input_dir)
    files = [x for x in files if os.path.isfile(os.path.join(input_dir, x))]
    files = [x for x in files if x.endswith('.xyz')]
    for input_filename in files:
        output_filename = input_filename.removesuffix('.xyz') + '_aligned.xyz'
        input_path = os.path.join(input_dir, input_filename)
        output_path = os.path.join(output_dir, output_filename)
        write_all_aligned(input_path, output_path, index_target=index_target, threshold=threshold, verbose=verbose)

In [9]:
results_9w_dir = '/misc/home/guest50/OAReactDiff/results/sample_selected_TS-20250709-SCAN-9w-various_models'
results_10w_dir = ''

In [10]:
files = os.listdir(results_9w_dir)

In [11]:
files

['WL1-TS784.xyz',
 'WL1-TS6029.xyz',
 'WL1-TS11193.xyz',
 'rcmconly_passerini-TS1206.xyz',
 'rcmconly_passerini-TS5435.xyz',
 'rcmconly_passerini-TS7394.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'rcmconly_passerini-TS15965.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS17605.xyz',
 'rcmconly_strecker-TS3978.xyz',
 'rcmconly_strecker-TS5528.xyz',
 'rcmconly_strecker-TS7950.xyz']

In [12]:
[x for x in files if os.path.isfile(os.path.join(results_9w_dir, x))]

['WL1-TS784.xyz',
 'WL1-TS6029.xyz',
 'WL1-TS11193.xyz',
 'rcmconly_passerini-TS1206.xyz',
 'rcmconly_passerini-TS5435.xyz',
 'rcmconly_passerini-TS7394.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'rcmconly_passerini-TS15965.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS17605.xyz',
 'rcmconly_strecker-TS3978.xyz',
 'rcmconly_strecker-TS5528.xyz',
 'rcmconly_strecker-TS7950.xyz']

In [13]:
[x for x in files if x.endswith('.xyz')]

['WL1-TS784.xyz',
 'WL1-TS6029.xyz',
 'WL1-TS11193.xyz',
 'rcmconly_passerini-TS1206.xyz',
 'rcmconly_passerini-TS5435.xyz',
 'rcmconly_passerini-TS7394.xyz',
 'rcmconly_passerini-TS13605.xyz',
 'rcmconly_passerini-TS15965.xyz',
 'rcmconly_passerini-TS17328.xyz',
 'rcmconly_passerini-TS17605.xyz',
 'rcmconly_strecker-TS3978.xyz',
 'rcmconly_strecker-TS5528.xyz',
 'rcmconly_strecker-TS7950.xyz']

In [14]:
os.makedirs('/misc/home/guest50/OAReactDiff/results/sample_selected_TS-20250709-SCAN-9w-various_models_aligned', exist_ok=True)